# 02 — Phase 2: Logit-Lens Trajectories

Per-item logit-difference curve across layers, for both arms.

**Phase transition** = the first layer at which the curve crosses zero, or `None`. The previous
implementation fell back to `argmax|diff|` whenever there was no sign change, which made the
reported detection rate 100% by construction. A curve that never crosses zero has no phase
transition, and reporting that is the informative answer.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
%load_ext autoreload
%autoreload 2

In [2]:
from circuit_conflict.utils import load_model
from circuit_conflict import dataset as D, metrics as M, pipeline as PL
import json

model = load_model()
admitted = D.load_prompts().query("passes_precondition")
lens = PL.run_logit_lens(model, admitted)
lens.to_csv(PL.RESULTS / "phase2" / "phase2_summary.csv", index=False)
lens.head()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2 into HookedTransformer
Loaded gpt2 on mps
  n_layers=12, n_heads=12, d_model=768, d_head=64


,item_id,category,arm,phase_transition_layer,final_diff
0,A_000,A,conflict,5.0,1.681410
1,A_002,A,conflict,1.0,0.102757
2,A_004,A,conflict,6.0,0.637641
3,A_006,A,conflict,11.0,-0.217160
4,A_008,A,conflict,9.0,-0.133970


## Phase-transition statistics (note the honest detection rate)

In [3]:
stats = {c: M.phase_transition_stats(
             lens.query("category == @c and arm == 'conflict'").phase_transition_layer.tolist())
         for c in ["A", "B", "C"]}
for c, s in stats.items():
    print(f"  {c}: detected {s['n_detected']}/{s['n_total']} "
          f"({s['detection_rate']:.0%})  mean layer {s['mean']}")
(PL.RESULTS / "phase2" / "phase_transition_stats.json").write_text(json.dumps(stats, indent=2))

  A: detected 33/40 (82%)  mean layer 5.121212121212121
  B: detected 21/23 (91%)  mean layer 5.809523809523809
  C: detected 19/25 (76%)  mean layer 4.2105263157894735


507